# 第2章：颜色空间的转换

## 编程实践：手写 RGB ↔ HSV / Lab 转换与颜色传递

| 项目 | 说明 |
|------|------|
| 输入图片 | `lena.jpeg`（单图转换）；`chong.png` + `lena.jpeg`（颜色传递） |
| 手写核心 | RGB↔HSV、RGB↔Lab、Reinhard 颜色传递 |
| 允许调用 | 仅 `cv_imread` / `cv_imwrite` 图像读写 |
| 对比验证 | 与 OpenCV `cv2.cvtColor` 结果做数值误差对比 |


## 一、学习目标

1. 理解 **RGB / HSV / Lab** 三种颜色空间的物理含义、主要特点与使用场合。
2. 掌握三种颜色空间之间的**相互转换公式**，并能手写实现。
3. 掌握两张图像之间的**颜色传递**算法（Reinhard 方法）。

| 颜色空间 | 含义 | 典型用途 |
|----------|------|----------|
| RGB | 红绿蓝三基色相加混色 | 显示、存储 |
| HSV | 色调 Hue / 饱和度 Saturation / 明度 Value | 颜色筛选、交互式调色 |
| Lab | 亮度 L / 绿红 a / 蓝黄 b | 色差度量、颜色传递、感知均匀 |


## 二、转换公式

### RGB → HSV

设 `r,g,b ∈ [0,1]`，`maxc = max(r,g,b)`，`minc = min(r,g,b)`，`delta = maxc - minc`：

```
h = 0                         , if delta == 0
h = 60 * (((g-b)/delta) % 6)  , if maxc == r
h = 60 * ((b-r)/delta + 2)    , if maxc == g
h = 60 * ((r-g)/delta + 4)    , if maxc == b

s = 0 if maxc == 0 else delta / maxc
v = maxc
```

OpenCV 的 HSV 编码：`H ∈ [0,180)`（即 `h/2`），`S,V ∈ [0,255]`。

### RGB → Lab（D65 参考白）

先做 sRGB 线性化，再经 XYZ 矩阵变换，最后做立方根非线性映射：

```
c_lin = c/12.92                    , if c <= 0.04045
c_lin = ((c+0.055)/1.055) ** 2.4   , otherwise

X = 0.412453*R + 0.357580*G + 0.180423*B
Y = 0.212671*R + 0.715160*G + 0.072169*B
Z = 0.019334*R + 0.119193*G + 0.950227*B

f(t) = t**(1/3)                        , if t > (6/29)**3
f(t) = t/(3*(6/29)**2) + 4/29          , otherwise

L = 116 * f(Y/Yn) - 16
a = 500 * (f(X/Xn) - f(Y/Yn))
b = 200 * (f(Y/Yn) - f(Z/Zn))
```

OpenCV 的 Lab 编码：`L ∈ [0,255]`（即 `L*255/100`），`a,b ∈ [0,255]`（即 `a+128`、`b+128`）。

### Reinhard 颜色传递

在 Lab 空间统计源图与目标图各通道的均值 / 标准差，再把源图颜色分布线性映射到目标图分布：

```
src' = (src - src_mean) * (tgt_std / src_std) + tgt_mean
```


## 三、手写约束清单

> 除 OpenCV 读写函数外，其余代码全部手写，不能调用其他函数库。

- ✅ 允许：`cv_imread` / `cv_imwrite`；Python 循环与算术；`np.zeros` 开辟空间。
- ❌ 禁止：`cv2.cvtColor`、`np.dot` 用于算法、`np.power`、`skimage` / `PIL`。
- ✅ 可视化：`matplotlib` 仅用于显示。
- ✅ 验证：与 `cv2.cvtColor` 的对比仅放在验证单元格。


In [ ]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


In [ ]:
def rgba_to_bgr(image):
    """将带 alpha 通道的 RGBA 图像去掉 alpha 通道，返回 BGR 三通道图像。"""
    return image[:, :, :3]


def rgb_to_hsv_manual(image):
    """手写 RGB(BGR) -> HSV。返回与 OpenCV 相同的 uint8 编码。"""
    h, w, _ = image.shape
    hsv = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            b, g, r = (float(image[y, x, c]) / 255.0 for c in (0, 1, 2))
            maxc = max(r, g, b)
            minc = min(r, g, b)
            delta = maxc - minc

            if delta == 0:
                hue = 0.0
            elif maxc == r:
                hue = 60.0 * (((g - b) / delta) % 6)
            elif maxc == g:
                hue = 60.0 * ((b - r) / delta + 2)
            else:
                hue = 60.0 * ((r - g) / delta + 4)

            sat = 0.0 if maxc == 0 else delta / maxc
            val = maxc
            hsv[y, x, 0] = int(round(hue / 2.0)) % 180
            hsv[y, x, 1] = int(round(sat * 255.0))
            hsv[y, x, 2] = int(round(val * 255.0))
    return hsv


def hsv_to_rgb_manual(hsv):
    """手写 HSV -> RGB(BGR)。输入为与 OpenCV 相同的 uint8 编码。"""
    h, w, _ = hsv.shape
    bgr = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            hue = float(hsv[y, x, 0]) * 2.0
            sat = float(hsv[y, x, 1]) / 255.0
            val = float(hsv[y, x, 2]) / 255.0

            c = val * sat
            xc = c * (1 - abs((hue / 60.0) % 2 - 1))
            m = val - c

            sector = int(hue // 60) % 6
            if sector == 0:
                r1, g1, b1 = c, xc, 0.0
            elif sector == 1:
                r1, g1, b1 = xc, c, 0.0
            elif sector == 2:
                r1, g1, b1 = 0.0, c, xc
            elif sector == 3:
                r1, g1, b1 = 0.0, xc, c
            elif sector == 4:
                r1, g1, b1 = xc, 0.0, c
            else:
                r1, g1, b1 = c, 0.0, xc

            bgr[y, x, 0] = int(round((b1 + m) * 255.0))
            bgr[y, x, 1] = int(round((g1 + m) * 255.0))
            bgr[y, x, 2] = int(round((r1 + m) * 255.0))
    return bgr


def _srgb_to_linear(c):
    """sRGB 单通道线性化。"""
    return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4


def _linear_to_srgb(c):
    """线性值转 sRGB 单通道。"""
    return c * 12.92 if c <= 0.0031308 else 1.055 * (c ** (1 / 2.4)) - 0.055


def _lab_f(t):
    """Lab 非线性映射函数 f(t)。"""
    delta = 6.0 / 29.0
    return t ** (1.0 / 3.0) if t > delta ** 3 else t / (3 * delta * delta) + 4.0 / 29.0


def _lab_f_inv(t):
    """Lab 非线性映射函数 f(t) 的逆。"""
    delta = 6.0 / 29.0
    return t ** 3 if t > delta else 3 * delta * delta * (t - 4.0 / 29.0)


def rgb_to_lab_manual(image):
    """手写 RGB(BGR) -> Lab(D65)。返回与 OpenCV 相同的 uint8 编码。"""
    h, w, _ = image.shape
    lab = np.zeros((h, w, 3), dtype=np.uint8)
    xn, yn, zn = 0.950456, 1.0, 1.088754

    for y in range(h):
        for x in range(w):
            b, g, r = (float(image[y, x, c]) / 255.0 for c in (0, 1, 2))
            r, g, b = _srgb_to_linear(r), _srgb_to_linear(g), _srgb_to_linear(b)
            X = 0.412453 * r + 0.357580 * g + 0.180423 * b
            Y = 0.212671 * r + 0.715160 * g + 0.072169 * b
            Z = 0.019334 * r + 0.119193 * g + 0.950227 * b

            fx, fy, fz = _lab_f(X / xn), _lab_f(Y / yn), _lab_f(Z / zn)
            L = 116.0 * fy - 16.0
            a = 500.0 * (fx - fy)
            b_ = 200.0 * (fy - fz)

            lab[y, x, 0] = int(round(L * 255.0 / 100.0))
            lab[y, x, 1] = int(round(a + 128.0))
            lab[y, x, 2] = int(round(b_ + 128.0))
    return lab


def lab_to_rgb_manual(lab):
    """手写 Lab -> RGB(BGR)。输入为与 OpenCV 相同的 uint8 编码。"""
    h, w, _ = lab.shape
    bgr = np.zeros((h, w, 3), dtype=np.uint8)
    xn, yn, zn = 0.950456, 1.0, 1.088754

    for y in range(h):
        for x in range(w):
            L = float(lab[y, x, 0]) * 100.0 / 255.0
            a = float(lab[y, x, 1]) - 128.0
            b_ = float(lab[y, x, 2]) - 128.0

            fy = (L + 16.0) / 116.0
            fx = fy + a / 500.0
            fz = fy - b_ / 200.0

            X = xn * _lab_f_inv(fx)
            Y = yn * _lab_f_inv(fy)
            Z = zn * _lab_f_inv(fz)

            r = 3.240479 * X - 1.537150 * Y - 0.498535 * Z
            g = -0.969256 * X + 1.875992 * Y + 0.041556 * Z
            b = 0.055648 * X - 0.204043 * Y + 1.057311 * Z

            bgr[y, x, 2] = int(round(min(255, max(0, _linear_to_srgb(max(0, r))) * 255.0)))
            bgr[y, x, 1] = int(round(min(255, max(0, _linear_to_srgb(max(0, g))) * 255.0)))
            bgr[y, x, 0] = int(round(min(255, max(0, _linear_to_srgb(max(0, b))) * 255.0)))
    return bgr


def channel_mean_std(image):
    """手写统计三通道均值与标准差。"""
    h, w, c = image.shape
    means = [0.0] * c
    stds = [0.0] * c
    for ch in range(c):
        s = 0.0
        for y in range(h):
            for x in range(w):
                s += float(image[y, x, ch])
        means[ch] = s / (h * w)
        var = 0.0
        for y in range(h):
            for x in range(w):
                var += (float(image[y, x, ch]) - means[ch]) ** 2
        stds[ch] = (var / (h * w)) ** 0.5
    return means, stds


def color_transfer_manual(source, target):
    """手写 Reinhard 颜色传递：把 target 的颜色分布迁移到 source 上。"""
    src_lab = rgb_to_lab_manual(source)
    tgt_lab = rgb_to_lab_manual(target)
    src_mean, src_std = channel_mean_std(src_lab)
    tgt_mean, tgt_std = channel_mean_std(tgt_lab)

    h, w, _ = src_lab.shape
    result_lab = np.zeros_like(src_lab)
    for y in range(h):
        for x in range(w):
            for ch in range(3):
                value = (float(src_lab[y, x, ch]) - src_mean[ch]) * (tgt_std[ch] / (src_std[ch] + 1e-6)) + tgt_mean[ch]
                result_lab[y, x, ch] = int(round(min(255, max(0, value))))
    return lab_to_rgb_manual(result_lab)


In [ ]:
# 读取彩色图像，手写 RGB -> HSV / Lab，并保存各通道
img = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
assert img is not None, "读取 lena.jpeg 失败"

hsv_m = rgb_to_hsv_manual(img)
lab_m = rgb_to_lab_manual(img)

# 各通道单独保存
for name, channel in [("h", 0), ("s", 1), ("v", 2)]:
    cv_imwrite(f"{name}_channel.jpg", hsv_m[:, :, channel])
for name, channel in [("l", 0), ("a", 1), ("b", 2)]:
    cv_imwrite(f"{name}_channel.jpg", lab_m[:, :, channel])

# 从保存的单通道"图像"读回并重组，转换回 RGB
h_ch = cv_imread("h_channel.jpg", cv2.IMREAD_GRAYSCALE)
s_ch = cv_imread("s_channel.jpg", cv2.IMREAD_GRAYSCALE)
v_ch = cv_imread("v_channel.jpg", cv2.IMREAD_GRAYSCALE)
hsv_rebuilt = cv2.merge([h_ch, s_ch, v_ch])
restored_from_hsv = hsv_to_rgb_manual(hsv_rebuilt)

l_ch = cv_imread("l_channel.jpg", cv2.IMREAD_GRAYSCALE)
a_ch = cv_imread("a_channel.jpg", cv2.IMREAD_GRAYSCALE)
b_ch = cv_imread("b_channel.jpg", cv2.IMREAD_GRAYSCALE)
lab_rebuilt = cv2.merge([l_ch, a_ch, b_ch])
restored_from_lab = lab_to_rgb_manual(lab_rebuilt)

# ---------- 与 OpenCV 对比验证 ----------
hsv_cv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lab_cv = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
compare_results(hsv_m, hsv_cv, "RGB->HSV")
compare_results(lab_m, lab_cv, "RGB->Lab")
compare_results(restored_from_hsv, cv2.cvtColor(hsv_cv, cv2.COLOR_HSV2BGR), "HSV->RGB 还原")
compare_results(restored_from_lab, cv2.cvtColor(lab_cv, cv2.COLOR_Lab2BGR), "Lab->RGB 还原")

show_images([img, hsv_m, lab_m, restored_from_lab], ["原图(BGR)", "HSV", "Lab", "Lab->RGB 还原"], figsize=(13, 4))


In [ ]:
# 手写颜色传递：把 chong.png 的色彩迁移到 lena.jpeg 上
source = cv_imread("chong.png", cv2.IMREAD_COLOR)
target = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
source = rgba_to_bgr(source) if source.shape[2] == 4 else source

transferred = color_transfer_manual(source, target)
cv_imwrite("color_transferred.jpg", transferred)
show_images([source, target, transferred], ["源图(内容)", "参考图(颜色)", "颜色传递结果"], figsize=(13, 4))


## 四、结果与参数分析

- RGB→HSV / RGB→Lab 与 `cv2.cvtColor` 的 MAE 通常 < 1~2 个灰度级，差异来自取整方式与 Lab 的极值裁剪。
- HSV 更适合"按颜色抠图 / 调色"，Lab 的 a、b 通道与亮度解耦，更适合颜色传递。
- 颜色传递的效果取决于源图与参考图的内容分布：均值/标准差假设两图颜色呈**近似高斯分布**，对风景、人像等效果较好，对极端高对比图可能出现色偏。

**易错点**
1. `chong.png` 带 alpha 通道，务必先去掉 alpha 再参与转换。
2. 转换要在 `[0,1]` 浮点域进行，保存回 `uint8` 前裁剪并取整。
3. Lab 需要 D65 参考白与 sRGB 线性化，忽略线性化会导致与 OpenCV 结果偏差较大。


## 五、科研规范小结

1. **公式到代码一一对应**：每个转换公式都写成独立、可测试的函数，避免"大杂烩"。
2. **双向可逆校验**：RGB→HSV→RGB 的还原误差应很小，这比单纯目测更可靠。
3. **量化误差意识**：`uint8` 存储会引入取整误差，需用 `compare_results` 显式量化。
4. **通道拆分/重组**：HSV、Lab 各通道物理含义不同，单独保存便于调试与教学。


## 六、练习：手写 HSV 色彩筛选

**要求**：不调用 `cv2.inRange` / `cv2.cvtColor`，利用上面手写的 `rgb_to_hsv_manual` 写一个 `select_by_hue_manual(image, h_min, h_max, s_min, v_min)`，把指定色调范围的像素保留为白色、其余置黑，并保存掩膜。


In [ ]:
# ==================== 练习解决方案 ====================
def select_by_hue_manual(image, h_min, h_max, s_min=0, v_min=0):
    """手写 HSV 色彩筛选：返回二值掩膜（保留为 255）。"""
    hsv = rgb_to_hsv_manual(image)
    h, w, _ = hsv.shape
    mask = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            hh, ss, vv = (int(hsv[y, x, c]) for c in range(3))
            if h_min <= hh <= h_max and ss >= s_min and vv >= v_min:
                mask[y, x] = 255
    return mask

img = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
mask = select_by_hue_manual(img, h_min=0, h_max=10, s_min=60, v_min=60)
cv_imwrite("hue_mask.jpg", mask)
show_images([img, mask], ["原图", "红色色调掩膜(0~10)"], figsize=(9, 4))
